# Experiment — Does `class_weight='balanced'` Improve Recall?

Same pipeline as the main capstone notebook, but the logistic regression is told to weight churners more heavily during training. Goal: see if recall on churners goes up (currently 48% in the baseline model).

## Step 1 — Load the Data, Pick Features and Target (same as baseline)

In [1]:
import pandas as pd

df = pd.read_csv("example-dataset-cleaned.csv")

features = ["tenure", "MonthlyCharges", "Contract", "InternetService"]
X = df[features]
y = df["Churn"].map({"Yes": 1, "No": 0})

X.head()

,tenure,MonthlyCharges,Contract,InternetService
0,1,29.85,Month-to-month,DSL
1,34,56.95,One year,DSL
2,2,53.85,Month-to-month,DSL
3,45,42.30,One year,DSL
4,2,70.70,Month-to-month,Fiber optic


## Step 2 — Encode Text Columns, Then Split (same as baseline)

In [2]:
X = pd.get_dummies(X, columns=["Contract", "InternetService"], drop_first=True)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape

((5634, 6), (1409, 6))

## Step 3 — Train the Logistic Regression WITH `class_weight='balanced'`

### What is `class_weight`?

It's a setting that tells the model: **"treat mistakes on this group as more (or less) important than mistakes on that group."**

**Why it's needed here:** normally, a model just tries to minimize its *total* number of mistakes — it doesn't care which class those mistakes are on. In this dataset, about 74% of customers stayed and only 26% churned. So the "cheapest" way for the model to look accurate is to lean toward predicting "stays" for almost everyone, since stayers are the majority. That's exactly the bias in the baseline model: it correctly predicted 88% of stayers, but only caught 48% of churners.

`class_weight='balanced'` fixes this by making mistakes on the minority class (churners) count more during training — roughly in inverse proportion to how rare that class is. So:
- Missing a churner now "hurts" the model's training score a lot more than it used to.
- The model shifts its decision boundary to lean more toward catching churners, even if that means occasionally misflagging a happy customer.

### Example 1 — the quiz analogy

Imagine grading a quiz where a rare question is worth 3x the points of a common one. You'd naturally start paying much more attention to the rare questions — even if it means occasionally missing an easy common one you'd normally get right. `class_weight='balanced'` is that grading rule, applied automatically based on how rare each class is in the data.

### Example 2 — a concrete customer

Take a customer who is genuinely borderline — say, the model is 45% confident they'll churn (just under the 50% cutoff to flag them).
- **Without** `class_weight`: the model predicts "stays," because that's the plain majority-vote guess, and a wrong guess on this customer only costs the model "1 mistake" either way.
- **With** `class_weight='balanced'`: a mistake on an actual churner is weighted more heavily during training, so the model's learned boundary shifts to catch more borderline cases like this one — it becomes more willing to flag "leave" even at lower confidence, since the cost of missing a real churner is now higher.

### Example 3 — how the weights are actually computed

`class_weight='balanced'` sets each class's weight as: `n_samples / (n_classes * count_of_that_class)`. With 4,139 stayers and 1,495 churners in the training data, that works out to:
- Stayers: weight ≈ 0.68 (weighted *down*, since they're common)
- Churners: weight ≈ 1.88 (weighted *up*, since they're rare — about 2.75x more than a stayer)

So every churner the model gets wrong during training counts almost 3x as much as getting a stayer wrong.

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## Step 4 — Predict on the Test Set

In [4]:
y_pred = model.predict(X_test)
model.score(X_test, y_test)

0.7224982256919801

## Step 5 — Evaluate With the Confusion Matrix

In [5]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, classification_report

print(confusion_matrix(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[720 315]
 [ 76 298]]
Recall: 0.7967914438502673
Precision: 0.48613376835236544
              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1035
           1       0.49      0.80      0.60       374

    accuracy                           0.72      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.72      0.74      1409



## Step 6 — Baseline vs. Balanced — Side by Side

| Metric | Baseline (`class_weight=None`) | Balanced (`class_weight='balanced'`) |
|---|---|---|
| Accuracy | 77% | 72% |
| Recall (churners) | 48% | **80%** |
| Precision (churners) | 59% | 49% |
| Missed churners (false negatives) | 195 | **76** |
| False alarms (false positives) | 123 | 315 |

**What actually happened:** recall jumped from 48% to **80%** — the model now catches 298 of the 374 real churners, missing only 76 (down from 195 missed). That's a huge win for the business goal of catching at-risk customers early.

The cost: precision dropped from 59% to 49%, and accuracy dropped from 77% to 72%. The model now raises far more false alarms (315 vs. 123) — nearly a third of customers who were never leaving get flagged as "at risk."

**Is this trade worth it?** Almost always yes, if the goal is proactive retention: a false alarm costs a cheap retention offer or a check-in call; a missed churner costs the customer's revenue outright. Going from missing 195 real churners to missing only 76 is a much stronger safety net, even though the model now "cries wolf" more often. If false alarms become too costly in practice (e.g. the retention team can't handle 500+ flagged customers), the decision threshold can be tuned to land somewhere between the baseline and this balanced version, rather than an all-or-nothing choice.

## Step 7 — Threshold Tuning

`class_weight='balanced'` changes how the model *learns*. Threshold tuning changes how we *read the model's output* — no retraining needed.

By default, `.predict()` flags a customer as "will churn" only if the model's confidence is above 50%. But that 50% cutoff is just a default, not a law of nature. Since missing a churner is usually more costly than a false alarm, we can lower the bar — e.g. flag anyone the model is even 20-30% confident about — to catch more real churners, at the cost of more false alarms.

`predict_proba()` gives the raw confidence score (0 to 1) for each customer, instead of a final yes/no. We can then apply our own cutoff to that score.

In [6]:
probs = model.predict_proba(X_test)[:, 1]  # confidence that each customer churns
probs[:10]

array([0.13153042, 0.84528925, 0.20603629, 0.78850185, 0.04357702,
       0.78853228, 0.78805713, 0.28134761, 0.01883171, 0.53417585])

In [7]:
from sklearn.metrics import recall_score, precision_score, accuracy_score, confusion_matrix

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]
results = []

for t in thresholds:
    y_pred_t = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
        "precision": precision_score(y_test, y_pred_t),
        "missed_churners (FN)": fn,
        "false_alarms (FP)": fp,
    })

import pandas as pd
pd.DataFrame(results)

,threshold,accuracy,recall,precision,missed_churners (FN),false_alarms (FP)
0,0.5,0.722498,0.796791,0.486134,76,315
1,0.4,0.694819,0.828877,0.458580,64,366
2,0.3,0.633073,0.938503,0.415385,23,494
3,0.2,0.577005,0.967914,0.382664,12,584
4,0.1,0.455642,0.991979,0.326872,3,764


### Reading the table

Lowering the threshold means: "flag someone as a churn risk even if the model is less confident." As the threshold drops from 0.5 toward 0.1:

- **Recall goes up** — the net gets wider, so more real churners get caught (missed churners / false negatives should shrink).
- **Precision goes down** — the net also scoops up more people who were never going to leave (false alarms / false positives should grow).
- **Accuracy** typically drops or stays flat — it's not the metric we're optimizing for here.

### How to pick a threshold in practice

There's no single "correct" threshold — it depends on the retention team's capacity and the real cost of each mistake:

- If the team can only realistically call **200 flagged customers a week**, pick whichever threshold produces close to that many flags.
- If a missed churner costs far more than a wasted call (e.g. losing $500/year in revenue vs. spending $10 on an offer), lean toward a **lower** threshold — accept more false alarms to avoid missing real churners.
- If false alarms actively annoy customers or overload staff, lean toward a threshold closer to 0.5.

A common approach: plot recall and precision across many thresholds (a "precision-recall curve") and pick the point where the trade-off best matches the business constraint, rather than testing a handful of round numbers like we did above.